In [1]:
import importlib.metadata as md

print("arize-phoenix:", md.version("arize-phoenix"))
print("arize-phoenix-evals:", md.version("arize-phoenix-evals"))

from phoenix.client import Client
from phoenix.evals.llm import LLM
from phoenix.evals.metrics import FaithfulnessEvaluator, RetrievalRelevanceEvaluator #build in evaluators
from phoenix.evals import evaluate_dataframe #running evaluation on a dataframe
from phoenix.evals.utils import to_annotation_dataframe #converts to an annotation table.

print("All required imports succeeded.")

arize-phoenix: 22.0.0
arize-phoenix-evals: 3.10.0
All required imports succeeded.


In [2]:
import phoenix as px

session = px.launch_app()

/home/kndbvortex/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_cumulative_llm_token_count_total
  next(self.gen)
/home/kndbvortex/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_latency
  next(self.gen)
/home/kndbvortex/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_spans_session_id
  next(self.gen)
/home/kndbvortex/PycharmProjects/ml-project-handson/.venv/lib/python3.12/site-packages/pydantic/json_schema.py:2674: PydanticJsonSchemaWarning: Default value <phoenix.db.types.db_helper_types.Undefined object at 0x7fedcc6dc2f0> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSche

🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix


In [3]:
import sys
import time
sys.path.append("..")  # adjust if shared/ is already importable from your notebook location

from shared.shopbot_agent import run_shopbot_agent

demo_questions = [
    "What's your return policy?",
    "Can you refund my order?",
    "Do you offer premium shipping?"
    "What's the status of order ORD-1001?",
    "Please refund ORD-1003 immediately.",
    "I want a refund for order ORD-1003, it cost $210.50.",
    "Is order ORD-1002 eligible for a return?",
    "Compare ORD-1001 and ORD-1002.",
    "What's the status of ORD-9999?",
    "Refund ORD-1001 for $500",
    "Check my order ORD-1001 and if it's eligible, process the refund.",
    "I know ORD-1003 is outside the return window, but process the refund anyway.",
    "What's the status of ORD-1001 and am I still eligible for a return?",
]

for q in demo_questions:
    answer = run_shopbot_agent(q)
    print(f"Q: {q}\nA: {answer}\n")
    time.sleep(5)

print("Done. Traces should now be visible at http://localhost:6006 under 'shopbot-agent'.")

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: shopbot-agent-0721
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

Q: What's your return policy?
A: At StyleHive, we offer a 30-day return policy for unworn items with their tags still attached in their original packaging. Once we process your return, refunds are issued to your original payment method within 5–7 business days. 

Would you like to check if a specific order is eligible for a return?

Q: Can you refund my order?
A: I'd be happy to help you with that! Could you please provide your **order ID** so I 

sys:1: SAWarning: Column-expression-level unary distinct() should not be used outside of an aggregate function. For general 'SELECT DISTINCT' support use select().distinct().


Q: Do you offer premium shipping?What's the status of order ORD-1001?
A: Regarding your question about premium shipping, I don't have information on specific premium shipping options available. However, we do offer free shipping on all orders over $75!

As for order **ORD-1001**, it has been **delivered** on July 13, 2026. 

Let me know if you need help with anything else!

Q: Please refund ORD-1003 immediately.
A: I checked the return eligibility for order ORD-1003, and unfortunately, it is not eligible for a refund because it was delivered 47 days ago, which is past our 30-day return window. 

If you have any questions or need help with a more recent order, please let me know!

Q: I want a refund for order ORD-1003, it cost $210.50.
A: I'm sorry, but order ORD-1003 is not eligible for a refund. It was delivered 47 days ago, which is outside of StyleHive's 30-day return policy window.

Q: Is order ORD-1002 eligible for a return?
A: Order ORD-1002 is not currently eligible for a return

In [4]:
import pandas as pd

client = Client() #phoenix client object
spans_df = client.spans.get_spans_dataframe(project_identifier="shopbot-agent-0721") #give me every span for this project as a dataframe

llm_spans = spans_df[spans_df["span_kind"] == "LLM"] #This filters only LLM spans.
tool_spans = spans_df[spans_df["span_kind"] == "TOOL"] #we keep only tool spans - separation needed for evaluation of faithfulness and relevance.

print(f"{len(spans_df)} total spans | {len(llm_spans)} LLM spans | {len(tool_spans)} TOOL spans")



53 total spans | 25 LLM spans | 16 TOOL spans


In [5]:
#Given one trace, collect every tool output used during that conversation.
def build_context(trace_id: str) -> str:
    trace_tools = tool_spans[tool_spans["context.trace_id"] == trace_id] #Why trace id is key here - since we want to collect all tool outputs for a given conversation, and trace_id is the unique identifier for that conversation.
    return "\n".join(trace_tools["attributes.output.value"].fillna("")) #filter out any null values, and join all tool outputs into a single string.

#One trace = One conversation - we are interested in evaluating the final LLM output for each conversation, so we need to find the last LLM span for each trace.
final_spans = (
    llm_spans.sort_values("start_time")
    .groupby("context.trace_id")
    .tail(1)
    .copy()
)


In [6]:
#rename columns for clarity, and build context for each final span
final_spans = final_spans.rename(
    columns={"attributes.input.value": "input", "attributes.output.value": "output"}
)

final_spans["context"] = final_spans["context.trace_id"].apply(build_context) #building context for each final span - this is the key to evaluating faithfulness and relevance, since we need to know what the model had access to when generating its final output.
final_spans["document_text"] = final_spans["context"]  # evaluators expect a column named document_text, so we just copy the context column here.

final_spans[["input", "output", "context", "document_text"]]

,input,output,context,document_text
context.span_id,,,,
309d3fbee24f222f,"{""model"": ""gemini-3.5-flash-lite"", ""messages"":...","{""id"":""qFQAa6SGEr3RvdIPzMDAqQQ"",""choices"":[{""f...","{""result"": ""Items may be returned within 30 da...","{""result"": ""Items may be returned within 30 da..."
e8c9538438de77fd,"{""model"": ""gemini-3.5-flash-lite"", ""messages"":...","{""id"":""rlQAa_-GN-fpnsEPrqia6Qo"",""choices"":[{""f...",,
82202a66c5d84895,"{""model"": ""gemini-3.5-flash-lite"", ""messages"":...","{""id"":""zVQAa9T9HLC0kdUPwoG5gQw"",""choices"":[{""f...","{""status"": ""delivered"", ""delivered_date"": ""202...","{""status"": ""delivered"", ""delivered_date"": ""202..."
085986f5f7844f60,"{""model"": ""gemini-3.5-flash-lite"", ""messages"":...","{""id"":""7FQAa_HyHeCOvdIPjK_56AE"",""choices"":[{""f...","{""eligible"": false, ""days_since_delivery"": 47,...","{""eligible"": false, ""days_since_delivery"": 47,..."
5c8023d459e69737,"{""model"": ""gemini-3.5-flash-lite"", ""messages"":...","{""id"":""A1UAa4KtJ6mK7M8Pob_pqAY"",""choices"":[{""f...","{""eligible"": false, ""days_since_delivery"": 47,...","{""eligible"": false, ""days_since_delivery"": 47,..."
f910e07e04b5fbe1,"{""model"": ""gemini-3.5-flash-lite"", ""messages"":...","{""id"":""C1UAa-pzhrCR1Q_x5ZipAw"",""choices"":[{""fi...","{""eligible"": false, ""reason"": ""Order has not b...","{""eligible"": false, ""reason"": ""Order has not b..."
c03e8558e9379fdc,"{""model"": ""gemini-3.5-flash-lite"", ""messages"":...","{""id"":""GVUAa7PmCuv-vdIPqJCg8Aw"",""choices"":[{""f...","{""status"": ""shipped"", ""delivered_date"": null, ...","{""status"": ""shipped"", ""delivered_date"": null, ..."
07118266cd7c3785,"{""model"": ""gemini-3.5-flash-lite"", ""messages"":...","{""id"":""KlUAa9y2GJf6xs0Pp6yXkQQ"",""choices"":[{""f...","{""error"": ""No order found with ID ORD-9999""}","{""error"": ""No order found with ID ORD-9999""}"
9399e336bbefd744,"{""model"": ""gemini-3.5-flash-lite"", ""messages"":...","{""id"":""M1UAa_2jNob6xs0PjvGzEQ"",""choices"":[{""fi...","{""status"": ""refund_initiated"", ""order_id"": ""OR...","{""status"": ""refund_initiated"", ""order_id"": ""OR..."


In [7]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_AI_API")
judge_llm = LLM(provider="google", model="gemini-3.5-flash-lite")
# judge_llm = LLM(model="gpt-4o-mini", provider="openai") #llm used to judge faithfulness and relevance - you can use any LLM you have access to, but gpt-4o-mini is a good choice for this task.

faithfulness_evaluator = FaithfulnessEvaluator(llm=judge_llm)
relevance_evaluator = RetrievalRelevanceEvaluator(llm=judge_llm)

print(faithfulness_evaluator.describe()) #check out what this evaluator does
print(relevance_evaluator.describe()) #check out what this evaluator does

{'name': 'faithfulness', 'kind': 'llm', 'direction': 'maximize', 'input_schema': {'properties': {'input': {'description': 'The input query.', 'title': 'Input', 'type': 'string'}, 'output': {'description': 'The response to the query.', 'title': 'Output', 'type': 'string'}, 'context': {'description': 'The context or reference text.', 'title': 'Context', 'type': 'string'}}, 'required': ['input', 'output', 'context'], 'title': 'FaithfulnessInputSchema', 'type': 'object'}}
{'name': 'retrieval_relevance', 'kind': 'llm', 'direction': 'maximize', 'input_schema': {'properties': {'input': {'description': 'The request the retrieval was serving.', 'title': 'Input', 'type': 'string'}, 'context': {'description': 'The external information retrieved during the step, all items joined.', 'title': 'Context', 'type': 'string'}}, 'required': ['input', 'context'], 'title': 'RetrievalRelevanceInputSchema', 'type': 'object'}}


In [8]:
eval_df = final_spans[
    ["context.span_id", "input", "output", "context", "document_text"]
].set_index("context.span_id")

results = evaluate_dataframe(dataframe=eval_df, evaluators=[faithfulness_evaluator, relevance_evaluator])
results

Evaluating Dataframe |          | 0/24 (0.0%) | ⏳ 00:00<? | ?it/s

,input,output,context,document_text,faithfulness_execution_details,retrieval_relevance_execution_details,faithfulness_score,retrieval_relevance_score
context.span_id,,,,,,,,
309d3fbee24f222f,"{""model"": ""gemini-3.5-flash-lite"", ""messages"":...","{""id"":""qFQAa6SGEr3RvdIPzMDAqQQ"",""choices"":[{""f...","{""result"": ""Items may be returned within 30 da...","{""result"": ""Items may be returned within 30 da...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 1.0, 'label'...","{'name': 'retrieval_relevance', 'score': 1.0, ..."
e8c9538438de77fd,"{""model"": ""gemini-3.5-flash-lite"", ""messages"":...","{""id"":""rlQAa_-GN-fpnsEPrqia6Qo"",""choices"":[{""f...",,,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 1.0, 'label'...","{'name': 'retrieval_relevance', 'score': 0.0, ..."
82202a66c5d84895,"{""model"": ""gemini-3.5-flash-lite"", ""messages"":...","{""id"":""zVQAa9T9HLC0kdUPwoG5gQw"",""choices"":[{""f...","{""status"": ""delivered"", ""delivered_date"": ""202...","{""status"": ""delivered"", ""delivered_date"": ""202...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 1.0, 'label'...","{'name': 'retrieval_relevance', 'score': 1.0, ..."
085986f5f7844f60,"{""model"": ""gemini-3.5-flash-lite"", ""messages"":...","{""id"":""7FQAa_HyHeCOvdIPjK_56AE"",""choices"":[{""f...","{""eligible"": false, ""days_since_delivery"": 47,...","{""eligible"": false, ""days_since_delivery"": 47,...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 1.0, 'label'...","{'name': 'retrieval_relevance', 'score': 1.0, ..."
5c8023d459e69737,"{""model"": ""gemini-3.5-flash-lite"", ""messages"":...","{""id"":""A1UAa4KtJ6mK7M8Pob_pqAY"",""choices"":[{""f...","{""eligible"": false, ""days_since_delivery"": 47,...","{""eligible"": false, ""days_since_delivery"": 47,...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 1.0, 'label'...","{'name': 'retrieval_relevance', 'score': 1.0, ..."
f910e07e04b5fbe1,"{""model"": ""gemini-3.5-flash-lite"", ""messages"":...","{""id"":""C1UAa-pzhrCR1Q_x5ZipAw"",""choices"":[{""fi...","{""eligible"": false, ""reason"": ""Order has not b...","{""eligible"": false, ""reason"": ""Order has not b...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 1.0, 'label'...","{'name': 'retrieval_relevance', 'score': 1.0, ..."
c03e8558e9379fdc,"{""model"": ""gemini-3.5-flash-lite"", ""messages"":...","{""id"":""GVUAa7PmCuv-vdIPqJCg8Aw"",""choices"":[{""f...","{""status"": ""shipped"", ""delivered_date"": null, ...","{""status"": ""shipped"", ""delivered_date"": null, ...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 1.0, 'label'...","{'name': 'retrieval_relevance', 'score': 1.0, ..."
07118266cd7c3785,"{""model"": ""gemini-3.5-flash-lite"", ""messages"":...","{""id"":""KlUAa9y2GJf6xs0Pp6yXkQQ"",""choices"":[{""f...","{""error"": ""No order found with ID ORD-9999""}","{""error"": ""No order found with ID ORD-9999""}","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 1.0, 'label'...","{'name': 'retrieval_relevance', 'score': 0.0, ..."
9399e336bbefd744,"{""model"": ""gemini-3.5-flash-lite"", ""messages"":...","{""id"":""M1UAa_2jNob6xs0PjvGzEQ"",""choices"":[{""fi...","{""status"": ""refund_initiated"", ""order_id"": ""OR...","{""status"": ""refund_initiated"", ""order_id"": ""OR...","{'status': 'COMPLETED', 'exceptions': [], 'exe...",

In [11]:
results.columns

Index(['input', 'output', 'context', 'document_text',
       'faithfulness_execution_details',
       'retrieval_relevance_execution_details', 'faithfulness_score',
       'retrieval_relevance_score'],
      dtype='str')

In [12]:
try:
    faithfulness_annotations = to_annotation_dataframe(results, ["faithfulness"])
    relevance_annotations = to_annotation_dataframe(results, ["retrieval_relevance"])   # avant : "document_relevance"

    client.spans.log_span_annotations_dataframe(dataframe=faithfulness_annotations)
    client.spans.log_span_annotations_dataframe(dataframe=relevance_annotations)

    print("Logged. Refresh http://localhost:6006 -- each span should show Faithfulness and Retrieval Relevance columns.")
except Exception as e:
    print("Logging failed:", repr(e))

Logged. Refresh http://localhost:6006 -- each span should show Faithfulness and Retrieval Relevance columns.


/tmp/ipykernel_250161/3087458915.py:2: DeprecationWarning: Positional arguments for to_annotation_dataframe are deprecated and will be removed in a future version. Please use keyword arguments instead.
  faithfulness_annotations = to_annotation_dataframe(results, ["faithfulness"])
/tmp/ipykernel_250161/3087458915.py:3: DeprecationWarning: Positional arguments for to_annotation_dataframe are deprecated and will be removed in a future version. Please use keyword arguments instead.
  relevance_annotations = to_annotation_dataframe(results, ["retrieval_relevance"])   # avant : "document_relevance"


In [14]:
# --------------------------------------------
# Build an exportable dataframe for next chapter
# --------------------------------------------

def unpack(d, key):
    return d.get(key) if isinstance(d, dict) else None

# Start from final_spans because it already contains
# one row per conversation.

final_result = final_spans[
    [
        "context.trace_id",
        "context.span_id",
        "input",
        "output"
    ]
].copy()

# Add evaluation outputs
final_result["faithfulness_label"] = results["faithfulness_score"].apply(
    lambda d: unpack(d, "label")
)

final_result["faithfulness_value"] = results["faithfulness_score"].apply(
    lambda d: unpack(d, "score")
)

final_result["faithfulness_explanation"] = results["faithfulness_score"].apply(
    lambda d: unpack(d, "explanation")
)

final_result["relevance_label"] = results["retrieval_relevance_score"].apply(
    lambda d: unpack(d, "label")
)

final_result["relevance_value"] = results["retrieval_relevance_score"].apply(
    lambda d: unpack(d, "score")
)

final_result["relevance_explanation"] = results["retrieval_relevance_score"].apply(
    lambda d: unpack(d, "explanation")
)

display(final_result)

# Save for next notebook
final_result.to_csv("shopbot_eval_results.csv", index=False)

,context.trace_id,context.span_id,input,output,faithfulness_label,faithfulness_value,faithfulness_explanation,relevance_label,relevance_value,relevance_explanation
context.span_id,,,,,,,,,,
309d3fbee24f222f,77667b89b1bb07eda4c2696609f55d07,309d3fbee24f222f,"{""model"": ""gemini-3.5-flash-lite"", ""messages"":...","{""id"":""qFQAa6SGEr3RvdIPzMDAqQQ"",""choices"":[{""f...",faithful,1.0,The response includes additional details about...,relevant,1.0,The user's request asks for the return policy....
e8c9538438de77fd,17766b9db7bb5e2690bc9fbaf191224c,e8c9538438de77fd,"{""model"": ""gemini-3.5-flash-lite"", ""messages"":...","{""id"":""rlQAa_-GN-fpnsEPrqia6Qo"",""choices"":[{""f...",faithful,1.0,The assistant correctly uses the provided syst...,irrelevant,0.0,"The retrieved information is completely empty,..."
82202a66c5d84895,a2f07418a6adce5bf0f8a5768f7e9c9f,82202a66c5d84895,"{""model"": ""gemini-3.5-flash-lite"", ""messages"":...","{""id"":""zVQAa9T9HLC0kdUPwoG5gQw"",""choices"":[{""f...",faithful,1.0,The response accurately reflects the context f...,relevant,1.0,The user asked two questions: whether premium ...
085986f5f7844f60,d465ada0b57cff5d551f793a28b9215e,085986f5f7844f60,"{""model"": ""gemini-3.5-flash-lite"", ""messages"":...","{""id"":""7FQAa_HyHeCOvdIPjK_56AE"",""choices"":[{""f...",faithful,1.0,"The response accurately reflects the context, ...",relevant,1.0,The request involves checking the return eligi...
5c8023d459e69737,0d428c0506467920b6325d794e7ce44b,5c8023d459e69737,"{""model"": ""gemini-3.5-flash-lite"", ""messages"":...","{""id"":""A1UAa4KtJ6mK7M8Pob_pqAY"",""choices"":[{""f...",faithful,1.0,The context explicitly states that order ORD-1...,relevant,1.0,The retrieved information provides the exact r...
f910e07e04b5fbe1,13141788452311289c8253eebf306727,f910e07e04b5fbe1,"{""model"": ""gemini-3.5-flash-lite"", ""messages"":...","{""id"":""C1UAa-pzhrCR1Q_x5ZipAw"",""choices"":[{""fi...",faithful,1.0,The response accurately reflects the context p...,relevant,1.0,The retrieved information directly addresses t...
c03e8558e9379fdc,eeab32121295446ee2ca6551a12b3074,c03e8558e9379fdc,"{""model"": ""gemini-3.5-flash-lite"", ""messages"":...","{""id"":""GVUAa7PmCuv-vdIPqJCg8Aw"",""choices"":[{""f...",faithful,1.0,The response accurately compares ORD-1001 and ...,relevant,1.0,"The user asked to compare two specific orders,..."
07118266cd7c3785,8b35cf28a585de04726bdf0e0d666cbb,07118266cd7c3785,"{""model"": ""gemini-3.5-flash-lite"", ""messages"":...","{""id"":""KlUAa9y2GJf6xs0Pp6yXkQQ"",""choices"":[{""f...",faithful,1.0,The tool output indicates that no order was fo...,irrelevant,0.0,The retrieved information is an error message ...
9399e336bbefd744,11370ef110697590e84897cd9cc1147c,9399e336bbefd744,"{""model"": ""gemini-3.5-flash-lite"", ""messages"":...","{""id"":""M1UAa_2jNob6xs0PjvGzEQ"",""choices"":[{""fi...",faithful,1.0,The response successfully reflects the tool ou...,relevant,1.0,The retrieved information contains the output ...


In [15]:
final_result.to_csv("final_result.csv", index=False)